# Investment Research Agent

AAI-520. We set this up as a small research desk instead of one long prompt.
An orchestrator plans the work and hands packets to three specialists:
news and sentiment, earnings and fundamentals, and market technicals
(moving averages and RSI). The walkthrough below is Microsoft.

AAI-520 Group 6: Sandeep Bhargava, Ashish Kumar, and Aishwarya Pareek.

GitHub: https://github.com/aishwaryashiva/aai520-investment-research-agent

Nothing here is a recommendation to buy or sell.


In [1]:
print("Desk")
print("- Research Orchestrator")
print("- News Sentiment Specialist")
print("- Earnings Fundamental Specialist")
print("- Market Technical Specialist")


Desk
- Research Orchestrator
- News Sentiment Specialist
- Earnings Fundamental Specialist
- Market Technical Specialist


## Agent Design and Workflows

We did not want one model to do the whole note. The orchestrator
(`InvestmentResearchAgent`) plans tool calls, routes each item, and is the
only one that writes the final brief. The reading is split:

| Agent | What we hand it |
| --- | --- |
| News Sentiment Specialist | Raw headlines. It runs the chain, then writes the tone note. |
| Earnings Fundamental Specialist | Ratios, the income statement, analyst targets, earnings headlines. |
| Market Technical Specialist | Price window, 20-day and 50-day SMAs, RSI(14), macro headlines. |

They do not call each other. The orchestrator keeps a handoff list so we
can see who got which packet. A coverage check can add a tool call when
news or prices come back empty. The brief is rewritten until the rubric
average clears 4.0.

```text
symbol
  -> retrieve_memory
  -> plan: prices, fundamentals, news, analyst targets
  -> optional follow-up call if news or prices come back empty
  -> prompt chain on headlines
  -> route each item to earnings, news, or market
  -> specialists write findings and risks
  -> draft
  -> evaluate
  -> revise, then evaluate again
  -> store a memory note the next plan must read
```

### Prompt chaining

Headlines move through five stages, and each stage keeps its output:

1. **Ingest** copies the raw articles and assigns ids.
2. **Preprocess** drops empty or duplicate titles.
3. **Classify** labels each headline as earnings, macro, product, legal,
   analyst, or other, and as positive, negative, or neutral.
4. **Extract** turns each headline into one citable statement.
5. **Summarize** writes a short brief from the counts and the extracted claims.

That chain lives on the news agent, not on the orchestrator. Ingest and
preprocess are plain Python. Classify, extract, and summarize go through
the same prompt interface as the other agents. Prompts are in
`aai520_research/prompts.py`.

### Routing

Each piece of content goes to one specialist.

| Content | Specialist |
| --- | --- |
| Price history | Market |
| Fundamentals, income statement, analyst snapshot | Earnings |
| Earnings headlines | Earnings |
| Macro headlines | Market |
| Product, legal, analyst, and other headlines | News |

Prices always go to the market agent. Statements and targets always go to
the earnings agent. Headlines follow the category the news agent just
assigned. The orchestrator is the one that makes those handoffs.

### Evaluator-optimizer

The generator writes a narrative first pass: a stance, four sections, two
supportive points, and one risk. That pass is scored on five dimensions.
The optimizer then fills the gaps the rubric names: a longer key-figures
list, a two-sided case, quantified catalysts, and a check against any prior
thesis. The loop stops when the mean score reaches 4.0 or after two
revisions. If a later draft scores lower, the better draft is the one that
is kept.

| Dimension | What a 5 requires |
| --- | --- |
| Evidence | At least four figures taken from the tool outputs |
| Coverage | Substantive thesis, market, fundamentals, and news sections |
| Balance | At least two bull points and two bear points |
| Actionability | A stance plus two catalysts that contain a number |
| Consistency | Stance agrees with return and news, and a prior thesis is addressed |

### Model provider

If `OPENAI_API_KEY` is set, the same prompts go to that model. If it is
not set, `HeuristicLLM` answers the task tags locally. We left the key off
for this run so the numbers stay tied to the Yahoo data. The score that
decides whether to revise is still our rubric either way.


## Setup

The next cell puts the project root on the path, clears the demo memory file,
and builds one agent. Later cells reuse that agent so the second run can see
the note written by the first.


In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "aai520_research").exists() and (ROOT.parent / "aai520_research").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aai520_research import InvestmentResearchAgent, MemoryStore
from aai520_research.planner import coverage_gap_steps
from aai520_research.prompts import CLASSIFY_SYSTEM, EVALUATE_SYSTEM, ROUTE_SYSTEM
from aai520_research.reporting import format_routes, format_steps

memory_path = ROOT / "data" / "memory" / "notebook_demo.json"
if memory_path.exists():
    memory_path.unlink()

agent = InvestmentResearchAgent(memory=MemoryStore(memory_path))
print("Project root:", ROOT)
print("Model provider:", agent.llm.name)
print("Tools:", ", ".join(sorted(agent.registry.names())))
print("Rubric threshold:", agent.score_threshold)


Project root: D:\Work\USD\AAI-520\Final Project
Model provider: heuristic
Tools: get_analyst_snapshot, get_company_news, get_fundamentals, get_income_statement, get_price_history, retrieve_memory
Rubric threshold: 4.0


### Prompts the workflows send

These are the instructions we send for classification, routing, and
evaluation. The `[TASK:...]` tag is how the local handlers pick a function.
A hosted model just reads the rest of the prompt.


In [3]:
print(CLASSIFY_SYSTEM)
print("\n" + "=" * 72 + "\n")
print(ROUTE_SYSTEM)
print("\n" + "=" * 72 + "\n")
print(EVALUATE_SYSTEM)


[TASK:classify_news]
You classify financial headlines.
Input JSON: {"items": [{"id", "title", "summary"}]}.
Return JSON: {"labels": [{"id", "category", "sentiment", "rationale"}]}.
category is one of earnings, macro, product, legal, analyst, other.
sentiment is one of positive, negative, neutral, judged by the likely
short-term implication for the share price.
Use earnings for results, guidance, margins, and EPS.
Use macro for the economy, rates, inflation, and policy.
Use product for products, launches, partnerships, and strategy.
Use legal for regulation, litigation, and investigations.
Use analyst for ratings, price targets, upgrades, and downgrades.
Keep each rationale to one sentence. Return JSON only.


[TASK:route_content]
You route one piece of research content to a specialist.
Input JSON: {"id", "kind", "text", "category", "sentiment"}.
Return JSON: {"route": "earnings"|"news"|"market", "reason": str}.
Send financial statements, valuation, and earnings headlines to earnings.
S

## First research run

We clear memory in the setup cell, so this first pass has nothing to
remember. The plan should be prices, fundamentals, news, and the analyst
snapshot. The income statement shows up on the second pass.


In [4]:
SYMBOL = "MSFT"
first = agent.run(SYMBOL)
print("Provider:", first.llm_name)
print("Memory notes before this run:", len(first.memory_context))
print("\nPlan")
print(format_steps(first.plan))
print("\nFollow-up tool calls from self-reflection")
print(format_steps(first.gap_steps))


Provider: heuristic
Memory notes before this run: 0

Plan
1. get_price_history
   Goal: Measure the recent price trend, volatility, and drawdown.
   Why: A stance needs the recent return, volatility, and drawdown.
   Arguments: {'symbol': 'MSFT', 'period': '1y'}
2. get_fundamentals
   Goal: Collect valuation, growth, profitability, and profile fields.
   Why: The earnings specialist needs current reported fundamentals.
   Arguments: {'symbol': 'MSFT'}
3. get_company_news
   Goal: Ingest recent headlines for the prompt-chaining workflow.
   Why: Headlines feed classification, extraction, and the news brief.
   Arguments: {'symbol': 'MSFT', 'limit': 8}
4. get_analyst_snapshot
   Goal: Collect the consensus target and recommendation.
   Why: The published target gives the brief an external benchmark.
   Arguments: {'symbol': 'MSFT'}

Follow-up tool calls from self-reflection
No steps.


### Tool results

Prices, fundamentals, and headlines are loaded through the tool registry.
The monthly closes below are the series behind the return, volatility, and
drawdown cited in the brief.


In [5]:
def tool_data(run, name):
    matches = [item for item in run.tool_results if item["tool"] == name and item["ok"]]
    return matches[-1]["data"] if matches else {}

prices = tool_data(first, "get_price_history")
fundamentals = tool_data(first, "get_fundamentals")
analyst = tool_data(first, "get_analyst_snapshot")
news = tool_data(first, "get_company_news")

print(f"Last price: {prices.get('last_price')} {prices.get('currency')}")
print(f"Window return: {prices.get('period_return')}")
print(f"Annualized volatility: {prices.get('annualized_volatility')}")
print(f"Max drawdown: {prices.get('max_drawdown')}")
print(f"Company: {fundamentals.get('short_name')} | {fundamentals.get('sector')}")
print(f"Revenue growth: {fundamentals.get('revenue_growth')}")
print(f"Profit margin: {fundamentals.get('profit_margin')}")
print(f"Trailing P/E: {fundamentals.get('trailing_pe')}")
print(f"Analyst target: {analyst.get('target_mean')} ({analyst.get('recommendation')})")
print(f"News source: {news.get('source')} | headlines: {len(news.get('items') or [])}")
print()
for item in news.get("items") or []:
    print(f"- {item.get('published') or 'undated'} | {item.get('publisher') or 'unknown'}")
    print(f"  {item['title']}")

pd.DataFrame(prices.get("monthly_closes") or [])


Last price: 529.05 USD
Window return: 0.03859724420426058
Annualized volatility: 0.32957382867386725
Max drawdown: -0.34498397568158834
Company: Microsoft Corporation | Technology
Revenue growth: 0.177
Profit margin: 0.40305
Trailing P/E: 29.473536
Analyst target: 599.23755 (strong_buy)
News source: yahoo-search | headlines: 8

- 2026-10-17 | Simply Wall St.
  Microsoft (MSFT) Is Facing Fresh Regulatory Pressure
- 2026-10-16 | Insider Monkey
  Microsoft (MSFT) Remains a High-Quality AI Leader
- 2026-10-16 | Insider Monkey
  Strong Azure Results Powered Microsoft Corporation’s (MSFT) Rally
- 2026-10-16 | Insider Monkey
  High Switching Costs Protect Microsoft Corporation (MSFT) in the AI Era
- 2026-10-16 | GuruFocus.com
  ETFs Are Buying Microsoft (MSFT) on Oct. 14
- 2026-10-16 | Simply Wall St.
  How Earnings Beat Will Impact Microsoft (MSFT) Stock
- 2026-10-15 | Stocktwits
  MSFT Stock Gets Slew Of Price Target Hikes Ahead Of Earnings – TD Cowen Sees ‘Higher Than Normal’ Azure Beat Po

,date,close
0,2025-10-31,513.58
1,2025-11-30,488.91
2,2025-12-31,480.57
3,2026-01-31,427.58
4,2026-02-28,391.15
5,2026-03-31,368.68
6,2026-04-30,406.13
7,2026-05-31,449.39
8,2026-06-30,372.32
9,2026-07-31,463.85


### Prompt chain artifacts

The counts below are the chain's own bookkeeping: how many headlines entered,
how many preprocess removed, and what classify, extract, and summarize produced.


In [6]:
chain = first.chain
print(f"Ingested: {len(chain['ingested'])}")
print(f"Dropped in preprocess: {chain['dropped']}")
print(f"Remaining: {len(chain['preprocessed'])}")
print("\nClassify")
for item in chain["classified"]:
    print(f"- [{item['category']}/{item['sentiment']}] {item['title']}")
    print(f"  {item['rationale']}")
print("\nExtract")
for fact in chain["facts"]:
    print(f"- ({fact['category']}, {fact['sentiment']}) {fact['statement']}")
print("\nSummarize")
print(chain["brief"]["summary"])
print("Sentiment counts:", chain["brief"]["sentiment_counts"])
print("Category counts:", chain["brief"]["category_counts"])


Ingested: 8
Dropped in preprocess: 0
Remaining: 8

Classify
- [legal/negative] Microsoft (MSFT) Is Facing Fresh Regulatory Pressure
  Terms in the headline map to legal and negative.
- [product/positive] Microsoft (MSFT) Remains a High-Quality AI Leader
  Terms in the headline map to product and positive.
- [other/positive] Strong Azure Results Powered Microsoft Corporation’s (MSFT) Rally
  Terms in the headline map to other and positive.
- [product/positive] High Switching Costs Protect Microsoft Corporation (MSFT) in the AI Era
  Terms in the headline map to product and positive.
- [other/neutral] ETFs Are Buying Microsoft (MSFT) on Oct. 14
  Terms in the headline map to other and neutral.
- [earnings/positive] How Earnings Beat Will Impact Microsoft (MSFT) Stock
  Terms in the headline map to earnings and positive.
- [analyst/positive] MSFT Stock Gets Slew Of Price Target Hikes Ahead Of Earnings – TD Cowen Sees ‘Higher Than Normal’ Azure Beat Potential
  Terms in the headline map to

### Routing and specialists

The route table is the orchestrator deciding who gets each item.
Under that, each specialist's own note. The draft is only allowed to use
those notes plus the tool figures.


In [7]:
print(format_routes(first.routes))
print()
for name, report in first.specialists.items():
    print(f"## {report.get('agent', name)}  (confidence {report['confidence']})")
    print(report.get("persona", ""))
    print("Findings:")
    for line in report["findings"]:
        print(" -", line)
    print("Risks:")
    for line in report["risks"]:
        print(" -", line)
    print()


prices [market_data] -> market: Structured market_data observations belong to the market specialist. Structured market_data observations belong to the market specialist.
fundamentals [fundamentals] -> earnings: Structured fundamentals observations belong to the earnings specialist. Structured fundamentals observations belong to the earnings specialist.
analyst [analyst] -> earnings: Structured analyst observations belong to the earnings specialist. Structured analyst observations belong to the earnings specialist.
n0 [news] -> news: The legal headline is assigned to the news specialist.
n1 [news] -> news: The product headline is assigned to the news specialist.
n2 [news] -> news: The other headline is assigned to the news specialist.
n3 [news] -> news: The product headline is assigned to the news specialist.
n4 [news] -> news: The other headline is assigned to the news specialist.
n5 [news] -> earnings: The earnings headline is assigned to the earnings specialist.
n6 [news] -> news: Th

### Evaluator-optimizer iterations

The first row is the narrative draft. The following row is the revision.
`passed` is true when the mean of the five scores is at least 4.0.


In [8]:
score_rows = []
for record in first.iterations:
    row = {"revision": record["revision"], "overall": record["overall"], "passed": record["passed"]}
    row.update(record["scores"])
    score_rows.append(row)
display(pd.DataFrame(score_rows))

print("Feedback on the first draft:")
for note in first.iterations[0]["feedback"]:
    print("-", note)
print("\n===== First draft =====\n")
print(first.iterations[0]["report_markdown"])
print("\n===== Final brief =====\n")
print(first.final_markdown)


,revision,overall,passed,evidence,coverage,balance,actionability,consistency
0,0,3.8,False,3,5,3,3,5
1,1,5.0,True,5,5,5,5,5


Feedback on the first draft:
- Evidence is thin. Cite at least four key figures from the tool outputs, including the period return and a valuation or target figure.
- The case is thin on one side. Give at least two bull points and two bear points.
- Catalysts are too vague. Name at least two catalysts that include a number from the evidence.

===== First draft =====

# MSFT - Investment Research Brief
**Company:** Microsoft Corporation
**Stance:** Constructive
**Revision:** 0

## Thesis
Microsoft Corporation (MSFT) screens as constructive on this pass. The 1y return is 3.9%, revenue growth is 17.7%, and the news sample has a net sentiment of 4.

## Market
The last price is USD 529.05 and the 1y total return is 3.9%. Annualized volatility over the window is 33.0% and the maximum drawdown is -34.5%. The last price is -4.5% from the window high of USD 553.72 and 51.5% from the window low of USD 349.20. Average daily volume over the window is 31,920,325 shares. The 20-day SMA is USD 520.22

## Agent Functions and Capabilities

### Planning

The plan above is what the orchestrator wrote with an empty memory. Each
step has a goal, a tool, arguments, and a reason. If a model plan skips a
core tool, `ensure_required_steps` puts it back. When a prior run left a
`next_run` note, those instructions are copied into the arguments.

### Dynamic tool use

Tools are registered by name. The agent calls only names on that list.
After the planned calls, `coverage_gap_steps` inspects the results. An empty
headline list produces a second `get_company_news` call with a larger limit.
Missing prices produce a second `get_price_history` call on a shorter window.
On this Microsoft pull the first news call already returned headlines, so
the follow-up list is empty. The next cell feeds the same check an empty
news result so we can see the extra call it would have made.


In [9]:
forced_gap = coverage_gap_steps(
    SYMBOL,
    [
        {
            "tool": "get_company_news",
            "ok": True,
            "data": {"items": []},
            "arguments": {"symbol": SYMBOL, "limit": 8},
            "error": None,
        },
        {
            "tool": "get_price_history",
            "ok": True,
            "data": {"last_price": prices.get("last_price")},
            "arguments": {"symbol": SYMBOL, "period": "1y"},
            "error": None,
        },
    ],
    start_id=len(first.plan) + 1,
)
print(format_steps(forced_gap))


5. get_company_news
   Goal: Retry headlines after an empty news sample.
   Why: Self-reflection: the news tool returned no articles, so the agent requests a larger sample before it writes the brief.
   Arguments: {'symbol': 'MSFT', 'limit': 20}


### Self-reflection

Two reflections run at the end of a pass.

The rubric reflection is the iteration table above. It says whether the brief
is grounded, complete, two-sided, actionable, and consistent with the evidence.

The memory reflection writes lessons, gaps, and a `next_run` block. The
important flag after this first run is `pull_income_statement`. The agent did
not open the financial statements, so the note tells the next planner to do
that and to challenge the stance it just stored.


In [10]:
import json

print(first.reflection["narrative"])
print("\nLessons")
for lesson in first.reflection["lessons"]:
    print("-", lesson)
print("\nGaps")
for gap in first.reflection["gaps"]:
    print("-", gap)
print("\nNext-run instructions")
print(json.dumps(first.reflection["next_run"], indent=2))


MSFT closed this run with a Constructive stance and an evaluator score of 5.00. On the next run, pull the income statement and compare margins with this thesis. The headline sample was large enough to score sentiment, but it is not a filing.

Lessons
- On the next run, pull the income statement and compare margins with this thesis.
- The headline sample was large enough to score sentiment, but it is not a filing.

Gaps
- The income statement was not reviewed, so the margin trend is unverified.

Next-run instructions
{
  "pull_income_statement": true,
  "news_count": 8,
  "focus": [
    "Verify the revenue and net-margin trend on the income statement.",
    "Challenge the prior Constructive stance against fresh prices and headlines."
  ]
}


### Learning across runs

We run Microsoft again on the same orchestrator without clearing memory.
The new plan should add `get_income_statement` because the first reflection
asked for it, and the first step's reason should mention the prior stance.
The earnings agent's section should now include the multi-year margin path.


In [11]:
second = agent.run(SYMBOL)
print("Plan on the second run")
print(format_steps(second.plan))
print("\nPrior stance remembered:", second.memory_context[-1]["stance"])
print("Income statement included:", second.evidence["had_income_statement"])
statement = second.evidence.get("income_statement") or {}
print("Statement periods:", statement.get("periods"))
print("Revenue:", statement.get("revenue"))
print("Net margin:", statement.get("net_margin"))
print("\n===== Second brief =====\n")
print(second.final_markdown)
print("\nMemory file:", memory_path)
print("Notes stored:", len(agent.memory.recall(SYMBOL)))


Plan on the second run
1. get_price_history
   Goal: Measure the recent price trend, volatility, and drawdown.
   Why: A stance needs the recent return, volatility, and drawdown. Memory focus: Verify the revenue and net-margin trend on the income statement; Challenge the prior Constructive stance against fresh prices and headlines.
   Arguments: {'symbol': 'MSFT', 'period': '1y'}
2. get_fundamentals
   Goal: Collect valuation, growth, profitability, and profile fields.
   Why: The earnings specialist needs current reported fundamentals.
   Arguments: {'symbol': 'MSFT'}
3. get_company_news
   Goal: Ingest recent headlines for the prompt-chaining workflow.
   Why: A prior reflection set the news sample size to 8.
   Arguments: {'symbol': 'MSFT', 'limit': 8}
4. get_analyst_snapshot
   Goal: Collect the consensus target and recommendation.
   Why: The published target gives the brief an external benchmark.
   Arguments: {'symbol': 'MSFT'}
5. get_income_statement
   Goal: Compare revenue an

## Evaluation and Iteration

On the first Microsoft pass the draft scored 3.8 and the revision scored
5.0. The first draft only cited two figures, named one risk, and used a
catalyst with no number in it. The revision notes in the brief are the list
of what that feedback changed.

The second pass is the part that carries over. `get_income_statement` is
missing from the first plan and present in the second, because the
reflection stored `pull_income_statement: true`. The prior-thesis check
quotes the earlier stance and says whether we kept it.

A few things we would still change: Yahoo search still mixes in some
loosely related headlines, and the income statement is a summary rather
than a full filing. The market agent's SMA and RSI readings are in the
market section of the brief.


In [12]:
comparison = pd.DataFrame(
    [
        {
            "run": "first",
            "revision": record["revision"],
            "overall": record["overall"],
            "passed": record["passed"],
        }
        for record in first.iterations
    ]
    + [
        {
            "run": "second",
            "revision": record["revision"],
            "overall": record["overall"],
            "passed": record["passed"],
        }
        for record in second.iterations
    ]
)
display(comparison)

first_tools = [step["tool"] for step in first.plan]
second_tools = [step["tool"] for step in second.plan]
print("Tools added because of memory:", sorted(set(second_tools) - set(first_tools)))
print("\nHandoffs on the first run")
for handoff in first.delegations:
    print(f"- {handoff['from']} -> {handoff['to']}: {handoff['task']}")


,run,revision,overall,passed
0,first,0,3.8,False
1,first,1,5.0,True
2,second,0,3.4,False
3,second,1,5.0,True


Tools added because of memory: ['get_income_statement']

Handoffs on the first run
- Research Orchestrator -> News Sentiment Specialist: Run the headline chain and prepare a sentiment brief.
- Research Orchestrator -> Market Technical Specialist: Read trend, 20-day and 50-day moving averages, and RSI.
- Research Orchestrator -> Earnings Fundamental Specialist: Read ratios, the income statement, and earnings headlines.
- Research Orchestrator -> News Sentiment Specialist: Write the sentiment note from the routed headlines.


The handoff list above is the part we cared about showing: the orchestrator
asks the news agent to chain the headlines, then sends the chart to the
market agent and the statements to the earnings agent, then asks the news
agent for the sentiment note. The comparison table is the two runs side
by side.
